In [0]:
# Create a dummy dataset of retail transactions
data = [
    (1, "Laptop", "Electronics", 1200, "NY"),
    (2, "Phone", "Electronics", 800, "CA"),
    (3, "Desk Chair", "Furniture", 250, "NY"),
    (4, "Monitor", "Electronics", 400, "TX"),
    (5, "Lamp", "Furniture", 50, "CA")
]

# Define the structure (schema) of our data
columns = ["transaction_id", "product", "category", "price", "state"]

# Create the distributed DataFrame
df = spark.createDataFrame(data, schema=columns)

# Preview the data
df.show()


In [0]:
# Filter for  Electronics and show the result
electronics_df = df.filter(df.category == "Electronics")
electronics_df.show()

In [0]:
# Filter for Furninture and show results
furniture_df = df.filter(df.category == "Furniture")
furniture_df.show()
# Filter for NY and show results
ny_df = df.filter(df.state == "NY")
ny_df.show()
# Filter for CA


In [0]:
# Filter for CA and only select specific columns
ca_df = df.filter(df.state == "CA").select("transaction_id", "product", "price")
ca_df.show()


In [0]:
ca_electronics_df = df.filter(df.state == "CA").select("product", "price")

ca_electronics_df.show()


In [0]:

# Import the sum function from PySpark
from pyspark.sql.functions import sum

# Group by state and calculate the total price for each state
state_spend_df = df.groupBy("state").agg(sum("price").alias("total_spent"))

# Execute and view the results
state_spend_df.show()

In [0]:
# Write the satae spend data as a managed Delta table
state_spend_df.write.format("delta").mode("overwrite").saveAsTable("default.state_spending")

# Confirm it saved by reading it right back from the storiage sytem
saved_df = spark.read.table("default.state_spending") 
saved_df.show()

In [0]:
# Create a new row for Texas
new_data = [(6, "Tablet", "Electronics", 300, "TX")]
new_df = spark.createDataFrame(new_data, schema=["transaction_id", "product", "category", "price", "state"])

# Append this new row to our existing Delta table
new_df.write.format("delta").mode("append").saveAsTable("default.sate_spending")

# Look at the update table
spark.read.table("default.state_spending").show()

In [0]:

# 1. Save our original 5-row dataset to a clean table named 'transactions'
df.write.format("delta").mode("overwrite").saveAsTable("default.transactions")

# 2. Append the new 6th row (the Texas tablet) to that same table
new_df.write.format("delta").mode("append").saveAsTable("default.transactions")

# 3. Read the table back to see all 6 rows together
spark.read.table("default.transactions").show()

In [0]:
# Check the version history of our new table
spark.sql("DESCRIBE HISTORY default.transactions").select("version", "timestamp", "operation").show()

In [0]:
# Travel back to version 0
v0_df = spark.read.format("delta").option("versionAsOf", 0).table("default.transactions")

# Show the results
v0_df.show()

In [0]:
# The midnight batch contains a price update for the laptop (ID 1) and a brand new keyboard (ID 7)
updates_data = [
    (1, "Laptop", "Electronics", 1100, "NY"), 
    (7, "Keyboard", "Electronics", 75, "TX")
]

updates_df = spark.createDataFrame(updates_data, schema=["transaction_id", "product", "category", "price", "state"])
updates_df.show()

In [0]:
from delta.tables import DeltaTable

# Access the target Delta table
target_table = DeltaTable.forName(spark, "default.transactions")

# Run the merge operation
target_table.alias("target") \
  .merge(updates_df.alias("source"), "target.transaction_id = source.transaction_id") \
  .whenMatchedUpdateAll() \
  .whenNotMatchedInsertAll() \
  .execute()

# View the final results
spark.read.table("default.transactions").show()

In [0]:
# Optimize the table files and Z-Ordr by state for faster lookups
spark.sql("OPTIMIZE default.transactions ZORDER BY (state)").show()


In [0]:
# A messy dataset containing duplicates and missing/null values
messy_data = [
    (2, "Phone", "Electronics", 800, "CA"),      # Exact duplicate of what's already in the table
    (8, "Desk", None, 150, "NY"),                # Missing category (None becomes Null)
    (9, "Mouse", "Electronics", None, "TX"),      # Missing price
    (10, None, None, None, None)                  # Corrupted row (all Nulls)
]

messy_df = spark.createDataFrame(messy_data, schema=["transaction_id", "product", "category", "price", "state"])
messy_df.show()

In [0]:
# Drop duplicates based on the unique transaction_id
deduped_df = messy_df.dropDuplicates(["transaction_id"])
deduped_df.show()

In [0]:
# Drop duplicates based on the unique transaction_id
deduped_df = messy_df.dropDuplicates(["transaction_id"])
deduped_df.show()

In [0]:
# Strategy A: Drop any row where EVERY single value is missing (like ID 10)
clean_rows_df = deduped_df.dropna(how="all")

# Strategy B: Fill missing categories with 'Unknown' and missing prices with 0
final_clean_df = clean_rows_df.fillna({"category": "Unknown", "price": 0})

final_clean_df.show()